In [1]:
# Install dependencies
!pip install pandas numpy


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [ ]:
RAW_PATH      = 'geotechnical_master_raw.csv'
OUTPUT_PATH   = 'geotechnical_cleaned.csv'
PHI_MIN, PHI_MAX = 0.0, 50.0
CU_MIN           = 0.0

In [ ]:
# Load the raw dataset
def load_raw(path):
    # Read the raw CSV and return a DataFrame
    df = pd.read_csv(path)
    print(f'Loaded {df.shape[0]} rows × {df.shape[1]} columns')
    return df

In [ ]:
df_raw = load_raw(RAW_PATH)
df_raw.head(3)

In [ ]:
def consolidate_duplicate_columns(df):
    # Merge paired Atterberg columns: primary takes precedence, alternate fills gaps
    pairs = [('LL', 'LL (%)'), ('PL', 'PL (%)'), ('PI', 'PI (%)')]
    df = df.copy()
    for primary, alternate in pairs:
        if alternate in df.columns:
            # Fill nulls in primary using alternate column values
            df[primary] = df[primary].combine_first(df[alternate])
            df.drop(columns=[alternate], inplace=True)
            print(f'  Merged "{alternate}" → "{primary}"')
    return df

In [ ]:
df = consolidate_duplicate_columns(df_raw)
print(f'Columns after consolidation: {df.shape[1]}')

In [ ]:
def solve_missing_ll(row):
    # Derive LL when PL and PI are both known
    if pd.isna(row['LL']) and pd.notna(row['PL']) and pd.notna(row['PI']):
        return row['PL'] + row['PI']
    return row['LL']

def solve_missing_pl(row):
    # Derive PL when LL and PI are both known
    if pd.isna(row['PL']) and pd.notna(row['LL']) and pd.notna(row['PI']):
        return row['LL'] - row['PI']
    return row['PL']

def solve_missing_pi(row):
    # Derive PI when LL and PL are both known
    if pd.isna(row['PI']) and pd.notna(row['LL']) and pd.notna(row['PL']):
        return row['LL'] - row['PL']
    return row['PI']

def apply_atterberg_solver(df):
    # Apply all three solvers in sequence and report fills made
    df = df.copy()
    before = df[['LL','PL','PI']].isna().sum()
    df['LL'] = df.apply(solve_missing_ll, axis=1)
    df['PL'] = df.apply(solve_missing_pl, axis=1)
    df['PI'] = df.apply(solve_missing_pi, axis=1)
    after = df[['LL','PL','PI']].isna().sum()
    filled = before - after
    print('Atterberg values solved (filled):')
    print(filled.to_string())
    return df

In [ ]:
df = apply_atterberg_solver(df)

In [ ]:
def flag_and_fill_non_plastic(df):
    # Identify NP rows: all three Atterberg columns are null
    df = df.copy()
    np_mask = df['LL'].isna() & df['PL'].isna() & df['PI'].isna()
    # Create binary plasticity flag (1 = plastic, 0 = non-plastic)
    df['is_plastic'] = (~np_mask).astype(int)
    # Set Atterberg values to 0 for NP soils
    df.loc[np_mask, ['LL', 'PL', 'PI']] = 0.0
    print(f'Non-plastic soils identified: {np_mask.sum()}')
    print(f'Plastic soils: {(~np_mask).sum()}')
    return df

In [ ]:
df = flag_and_fill_non_plastic(df)
print(f'\nis_plastic value counts:')
print(df['is_plastic'].value_counts().to_string())

In [ ]:
def filter_phi(df, phi_min, phi_max):
    # Remove rows where Phi_deg is outside valid geotechnical range
    before = len(df)
    # Keep rows where Phi_deg is null (will be handled downstream) or within bounds
    mask = df['Phi_deg'].isna() | df['Phi_deg'].between(phi_min, phi_max)
    df = df[mask].copy()
    removed = before - len(df)
    print(f'Rows removed (Phi_deg out of [{phi_min}, {phi_max}]): {removed}')
    return df

def filter_cu(df, cu_min):
    # Remove rows where Cu_kPa is negative
    before = len(df)
    mask = df['Cu_kPa'].isna() | (df['Cu_kPa'] >= cu_min)
    df = df[mask].copy()
    removed = before - len(df)
    print(f'Rows removed (Cu_kPa < {cu_min}): {removed}')
    return df

def apply_physical_filters(df, phi_min, phi_max, cu_min):
    # Apply all physical bound filters and return cleaned DataFrame
    before = len(df)
    df = filter_phi(df, phi_min, phi_max)
    df = filter_cu(df, cu_min)
    print(f'Total rows removed by physical filtering: {before - len(df)}')
    print(f'Rows remaining: {len(df)}')
    return df

In [ ]:
df = apply_physical_filters(df, PHI_MIN, PHI_MAX, CU_MIN)

In [ ]:
def summarise_cleaned(df):
    # Print shape, key column null counts, and target variable stats
    print(f'Final shape: {df.shape}')
    print()
    key_cols = ['LL', 'PL', 'PI', 'is_plastic', 'Phi_deg', 'Cu_kPa']
    print('Null counts for key columns:')
    print(df[key_cols].isnull().sum().to_string())
    print()
    print('Target variable statistics:')
    print(df[['Phi_deg', 'Cu_kPa']].describe().round(3).to_string())

In [ ]:
summarise_cleaned(df)
df[['LL','PL','PI','is_plastic','Phi_deg','Cu_kPa']].head(8)

In [ ]:
def export_csv(df, path):
    # Write cleaned DataFrame to CSV without row index
    df.to_csv(path, index=False)
    print(f'Exported → {path}  ({df.shape[0]} rows × {df.shape[1]} cols)')

In [ ]:
export_csv(df, OUTPUT_PATH)